In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# 31. OpenTelemetry LLM Observability with Gradio

This Kaggle notebook reuses the Grafana OTLP setup from `14-opentelemetry-llm-observability-e5.ipynb` and replaces the visualization layer with a Gradio dashboard launched through Gradio's built-in public tunnel.

What this notebook does:
- runs a GGUF llama.cpp server through `llamatelemetry`
- exports traces and metrics to Grafana OTLP
- captures the same spans locally for notebook-side analysis
- launches a separate Gradio observability dashboard in a new browser tab using `share=True`


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import subprocess

print("=" * 70)
print("KAGGLE GPU CHECK")
print("=" * 70)
result = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print(result.stdout)


KAGGLE GPU CHECK
0, Tesla T4, 15360 MiB, 14913 MiB
1, Tesla T4, 15360 MiB, 14913 MiB



In [6]:
print("📦 Installing dependencies...")


!pip install -q --no-cache-dir --force-reinstall git+https://github.com/llamatelemetry/llamatelemetry.git@v0.1.1 > /dev/null

!pip install -q pyarrow pandas numpy scipy huggingface_hub

# Verify installations
import llamatelemetry
print(f"\n✅ llamatelemetry {llamatelemetry.__version__} installed")


📦 Installing dependencies...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2026.3.0 which is incompatible.
datasets 4.8.3 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
dopamine-

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  



🎯 llamatelemetry v0.1.1 First-Time Setup - Kaggle 2× T4 Multi-GPU

🎮 GPU Detected: Tesla T4 (Compute 7.5)
  ✅ Tesla T4 detected - Perfect for llamatelemetry v0.1.1!
🌐 Platform: Kaggle

📦 Downloading Kaggle 2× T4 binaries (~961 MB)...
    Features: FlashAttention + Tensor Cores + Multi-GPU tensor-split

➡️  Attempt 1: HuggingFace (llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz)
📥 Downloading v0.1.1 from HuggingFace Hub...
   Repo: waqasm86/llamatelemetry-binaries
   File: v0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz


v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/1.40G [00:00<?, ?B/s]

v0.1.1/llamatelemetry-v0.1.1-cuda12-kagg(…):   0%|          | 0.00/130 [00:00<?, ?B/s]

🔐 Verifying SHA256 checksum...
   ✅ Checksum verified
📦 Extracting llamatelemetry-v0.1.1-cuda12-kaggle-t4x2.tar.gz...
Found 21 files in archive
Extracted 21 files to /root/.cache/llamatelemetry/extract_0.1.1
✅ Extraction complete!
  Found bin/ and lib/ under /root/.cache/llamatelemetry/extract_0.1.1/llamatelemetry-v0.1.1-cuda12-kaggle-t4x2
  Copied 13 binaries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12
  Copied 2 libraries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/lib
✅ Binaries installed successfully!


✅ llamatelemetry 0.1.1 installed


In [7]:
!pip install -q gradio plotly opentelemetry-api==1.37.0 opentelemetry-sdk==1.37.0 opentelemetry-proto==1.37.0 \
  opentelemetry-exporter-otlp-proto-common==1.37.0 opentelemetry-exporter-otlp-proto-http==1.37.0 \
  rich==13.9.4   --upgrade-strategy=only-if-needed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.7/65.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.9/131.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 16.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.3 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.


In [ ]:
#from google.cloud import bigquery
#bigquery_client = bigquery.Client(project='YOUR PROJECT ID')

In [8]:
# Step 1: Setup Secrets (Kaggle Secrets)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()

# Grafana OTLP
GRAFANA_OTLP_ENDPOINT = secrets.get_secret("GRAFANA_OTLP_ENDPOINT").rstrip("/")
GRAFANA_OTLP_BASIC_B64 = secrets.get_secret("GRAFANA_OTLP_BASIC_B64")
GRAFANA_OTLP_INSTANCE_ID = secrets.get_secret("GRAFANA_OTLP_INSTANCE_ID")
GRAFANA_OTLP_TOKEN = secrets.get_secret("GRAFANA_OTLP_TOKEN")

# HuggingFace
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)

# Export OTel env vars for SDK auto-config (explicit v1 paths)
os.environ["OTEL_EXPORTER_OTLP_PROTOCOL"] = "http/protobuf"
os.environ["OTEL_EXPORTER_OTLP_LOGS_ENDPOINT"] = f"{GRAFANA_OTLP_ENDPOINT}/v1/logs"
os.environ["OTEL_EXPORTER_OTLP_TRACES_ENDPOINT"] = f"{GRAFANA_OTLP_ENDPOINT}/v1/traces"
os.environ["OTEL_EXPORTER_OTLP_METRICS_ENDPOINT"] = f"{GRAFANA_OTLP_ENDPOINT}/v1/metrics"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic%20{GRAFANA_OTLP_BASIC_B64}"
os.environ["OTEL_TRACES_EXPORTER"] = "otlp"
os.environ["OTEL_METRICS_EXPORTER"] = "otlp"

print("Grafana OTLP endpoint configured:", GRAFANA_OTLP_ENDPOINT)


Grafana OTLP endpoint configured: https://otlp-gateway-prod-us-east-0.grafana.net/otlp


In [9]:
# Step 2: OpenTelemetry Setup (Grafana OTLP + local memory export)
import logging
from opentelemetry import metrics, trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.http.metric_exporter import OTLPMetricExporter

logging.getLogger().setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry").setLevel(logging.CRITICAL)
logging.getLogger("opentelemetry").propagate = False

try:
    trace.get_tracer_provider().shutdown()
except Exception:
    pass

try:
    metrics.get_meter_provider().shutdown()
except Exception:
    pass

resource = Resource.create({
    "service.name": "llamatelemetry-gradio-observability",
    "service.version": "0.1.1",
    "service.instance.id": "kaggle-gradio-dashboard",
    "deployment.environment": "kaggle-notebook",
    "host.name": "kaggle-gpu-0",
    "gpu.model": "Tesla T4",
})

tracer_provider = TracerProvider(resource=resource)
span_exporter = OTLPSpanExporter(
    endpoint=f"{GRAFANA_OTLP_ENDPOINT}/v1/traces",
    headers={
        "Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}",
        "Content-Type": "application/x-protobuf",
    },
)
tracer_provider.add_span_processor(BatchSpanProcessor(span_exporter))
memory_exporter = InMemorySpanExporter()
tracer_provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer(__name__)

metric_exporter = OTLPMetricExporter(
    endpoint=f"{GRAFANA_OTLP_ENDPOINT}/v1/metrics",
    headers={"Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}"},
)
metric_reader = PeriodicExportingMetricReader(metric_exporter, export_interval_millis=5000)
meter_provider = MeterProvider(resource=resource, metric_readers=[metric_reader])
metrics.set_meter_provider(meter_provider)
meter = metrics.get_meter(__name__)

request_counter = meter.create_counter(
    name="llm.requests.total",
    description="Total number of LLM requests",
    unit="1",
)
latency_histogram = meter.create_histogram(
    name="llm.request.duration",
    description="LLM request latency",
    unit="ms",
)
token_histogram = meter.create_histogram(
    name="llm.tokens.total",
    description="Token usage per request",
    unit="{token}",
)

with tracer.start_as_current_span("grafana.sanity") as span:
    span.set_attribute("check", "ok")

print("OpenTelemetry providers initialized for Grafana + local dashboard")


OpenTelemetry providers initialized for Grafana + local dashboard


In [10]:
# Step 3: Download GGUF model and start llama.cpp server
import os
import torch
from huggingface_hub import hf_hub_download
from llamatelemetry.server import ServerManager

os.makedirs("/kaggle/working/models", exist_ok=True)

model_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-3B-Instruct-GGUF",
    filename="Qwen2.5-3B-Instruct-Q4_K_M.gguf",
    local_dir="/kaggle/working/models",
)
print(f"Model downloaded: {model_path}")

print(f"Detected {torch.cuda.device_count()} GPU(s)")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

server = ServerManager(server_url="http://127.0.0.1:8080")
server.start_server(
    model_path=model_path,
    gpu_layers=99,
    tensor_split="1.0",
    port=8080,
    host="127.0.0.1",
    ctx_size=4096,
    batch_size=512,
)

print("llama.cpp server ready on http://127.0.0.1:8080")


Qwen2.5-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

Model downloaded: /kaggle/working/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf
Detected 1 GPU(s)
GPU 0: Tesla T4
GPU Check:
  Platform: kaggle
  GPU: Tesla T4
  Compute Capability: 7.5
  Status: ✓ Compatible
  Command: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server -m /kaggle/working/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf --host 127.0.0.1 --port 8080 -ngl 99 -c 4096 --parallel 1 -b 512 -ub 128 --tensor-split 1.0
Starting llama-server...
  Executable: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server
  Model: Qwen2.5-3B-Instruct-Q4_K_M.gguf
  GPU Layers: 99
  Context Size: 4096
  Server URL: http://127.0.0.1:8080
Waiting for server to be ready...... ✓ Ready in 3.0s
llama.cpp server ready on http://127.0.0.1:8080


In [12]:
# Step 4: Instrumented inference helpers
import random
import pandas as pd
import time
from opentelemetry.trace import Status, StatusCode
from llamatelemetry.api import LlamaCppClient

class InstrumentedLLMClient:
    def __init__(self, base_url: str, tracer):
        self.client = LlamaCppClient(base_url)
        self.tracer = tracer

    def chat_completion(self, messages: list, **kwargs):
        model = kwargs.get("model", "unknown")
        max_tokens = kwargs.get("max_tokens", 100)
        temperature = kwargs.get("temperature", 0.7)

        with self.tracer.start_as_current_span(
            name=f"llm.chat.{model}",
            kind=trace.SpanKind.CLIENT,
        ) as span:
            try:
                span.set_attribute("llm.system", "llama.cpp")
                span.set_attribute("llm.model", model)
                span.set_attribute("llm.request.max_tokens", max_tokens)
                span.set_attribute("llm.request.temperature", temperature)
                span.set_attribute("llm.request.messages", len(messages))

                start_time = time.time()
                response = self.client.chat.completions.create(messages=messages, **kwargs)
                latency_ms = (time.time() - start_time) * 1000

                finish_reason = response.choices[0].finish_reason
                content = response.choices[0].message.content
                span.set_attribute("llm.response.finish_reason", finish_reason)
                span.set_attribute("llm.response.length", len(content))

                request_counter.add(
                    1,
                    attributes={"model": model, "finish_reason": finish_reason, "status": "success"},
                )
                latency_histogram.record(latency_ms, attributes={"model": model, "status": "success"})

                if hasattr(response, "usage"):
                    input_tokens = getattr(response.usage, "prompt_tokens", 0)
                    output_tokens = getattr(response.usage, "completion_tokens", 0)
                    span.set_attribute("llm.usage.input_tokens", input_tokens)
                    span.set_attribute("llm.usage.output_tokens", output_tokens)
                    token_histogram.record(input_tokens, attributes={"model": model, "token_type": "input"})
                    token_histogram.record(output_tokens, attributes={"model": model, "token_type": "output"})

                span.set_status(Status(StatusCode.OK))
                return response
            except Exception as exc:
                span.set_status(Status(StatusCode.ERROR, str(exc)))
                span.record_exception(exc)
                request_counter.add(
                    1,
                    attributes={
                        "model": model,
                        "status": "error",
                        "error_type": type(exc).__name__,
                    },
                )
                raise

llm = InstrumentedLLMClient("http://127.0.0.1:8080", tracer)

def generate_sample_requests(batch_size: int = 6):
    prompts = [
        "Explain transformer architecture in simple terms.",
        "What is GGUF and why is it useful for local inference?",
        "Describe KV cache behavior in llama.cpp.",
        "Why does quantization reduce memory usage?",
        "How does OpenTelemetry tracing help with LLM observability?",
        "Compare latency and throughput for batch inference.",
    ]

    outputs = []
    with tracer.start_as_current_span("llm.batch.requests"):
        for i in range(batch_size):
            prompt = prompts[i % len(prompts)]
            response = llm.chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=random.randint(64, 160),
                temperature=round(random.uniform(0.4, 0.9), 2),
            )
            outputs.append(
                {
                    "prompt": prompt,
                    "response": response.choices[0].message.content,
                    "finish_reason": response.choices[0].finish_reason,
                }
            )
            time.sleep(0.4)

    tracer_provider.force_flush()
    return pd.DataFrame(outputs)

sample_df = generate_sample_requests(batch_size=6)
sample_df.head()


,prompt,response,finish_reason
0,Explain transformer architecture in simple terms.,Sure! Let's break down the Transformer archite...,length
1,What is GGUF and why is it useful for local in...,"GGUF stands for ""Generalized Gradient Update F...",length
2,Describe KV cache behavior in llama.cpp.,"In Llama.cpp, which is part of the LLaMA (Lang...",length
3,Why does quantization reduce memory usage?,Quantization reduces memory usage by represent...,length
4,How does OpenTelemetry tracing help with LLM o...,OpenTelemetry tracing is a powerful tool for m...,length


In [13]:
# Step 5: Prepare Gradio dashboard data model
import plotly.express as px
import plotly.graph_objects as go

def spans_to_dataframe():
    rows = []
    for span in memory_exporter.get_finished_spans():
        attrs = span.attributes or {}
        rows.append(
            {
                "span_name": span.name,
                "status": span.status.status_code.name,
                "start_time": pd.to_datetime(span.start_time, unit="ns"),
                "duration_ms": (span.end_time - span.start_time) / 1_000_000,
                "model": attrs.get("llm.model", "unknown"),
                "input_tokens": attrs.get("llm.usage.input_tokens", 0),
                "output_tokens": attrs.get("llm.usage.output_tokens", 0),
                "total_tokens": attrs.get("llm.usage.input_tokens", 0) + attrs.get("llm.usage.output_tokens", 0),
                "temperature": attrs.get("llm.request.temperature", None),
                "max_tokens": attrs.get("llm.request.max_tokens", None),
                "finish_reason": attrs.get("llm.response.finish_reason", "n/a"),
            }
        )

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("start_time").reset_index(drop=True)
    return df

def build_dashboard_state():
    df = spans_to_dataframe()

    if df.empty:
        empty_fig = go.Figure().update_layout(title="No spans captured yet")
        return (
            "### No telemetry spans yet\nRun sample requests from the dashboard.",
            empty_fig,
            empty_fig,
            pd.DataFrame(columns=["prompt", "response", "finish_reason"]),
            pd.DataFrame(columns=["start_time", "span_name", "duration_ms", "model", "total_tokens", "status"]),
        )

    recent_requests = sample_df.tail(10).copy() if 'sample_df' in globals() else pd.DataFrame()
    latency_fig = px.histogram(
        df,
        x="duration_ms",
        nbins=20,
        title="Latency Distribution (ms)",
        color="status",
    )
    token_fig = px.line(
        df,
        x="start_time",
        y=["input_tokens", "output_tokens", "total_tokens"],
        title="Token Usage Over Time",
        markers=True,
    )

    summary = (
        f"### LLM Observability Summary\n"
        f"- Captured spans: **{len(df)}**\n"
        f"- Mean latency: **{df['duration_ms'].mean():.2f} ms**\n"
        f"- Total tokens observed: **{int(df['total_tokens'].sum())}**\n"
        f"- Models seen: **{', '.join(sorted(df['model'].astype(str).unique()))}**\n"
        f"- Grafana OTLP export: **enabled**"
    )

    table = df[["start_time", "span_name", "duration_ms", "model", "total_tokens", "status"]].tail(25)
    return summary, latency_fig, token_fig, recent_requests, table

build_dashboard_state()[0]


'### LLM Observability Summary\n- Captured spans: **15**\n- Mean latency: **2822.43 ms**\n- Total tokens observed: **1795**\n- Models seen: **unknown**\n- Grafana OTLP export: **enabled**'

In [14]:
# Step 6: Launch a separate Gradio dashboard with built-in tunneling
import gradio as gr
from IPython.display import HTML, display

def refresh_dashboard():
    return build_dashboard_state()

def run_batch_and_refresh(batch_size):
    global sample_df
    sample_df = generate_sample_requests(batch_size=int(batch_size))
    return build_dashboard_state()

with gr.Blocks(title="LlamaTelemetry Observability Dashboard") as demo:
    gr.Markdown(
        "# LlamaTelemetry Gradio Observability Dashboard\n"
        "This dashboard is backed by local spans collected in Kaggle while the same telemetry is exported to Grafana OTLP."
    )
    with gr.Row():
        batch_size = gr.Slider(minimum=1, maximum=12, value=6, step=1, label="Sample request batch size")
        run_btn = gr.Button("Generate sample traffic", variant="primary")
        refresh_btn = gr.Button("Refresh dashboard")
    summary = gr.Markdown()
    with gr.Row():
        latency_plot = gr.Plot(label="Latency")
        token_plot = gr.Plot(label="Tokens")
    recent_requests = gr.Dataframe(label="Recent sample requests")
    span_table = gr.Dataframe(label="Recent spans")

    demo.load(refresh_dashboard, inputs=None, outputs=[summary, latency_plot, token_plot, recent_requests, span_table])
    run_btn.click(run_batch_and_refresh, inputs=[batch_size], outputs=[summary, latency_plot, token_plot, recent_requests, span_table])
    refresh_btn.click(refresh_dashboard, inputs=None, outputs=[summary, latency_plot, token_plot, recent_requests, span_table])

app, local_url, share_url = demo.launch(
    share=True,
    inline=False,
    prevent_thread_lock=True,
    quiet=False,
    show_api=False,
)

display(HTML(f'<a href="{share_url}" target="_blank">Open the Gradio observability dashboard in a new tab</a>'))
print("Local URL:", local_url)
print("Public share URL:", share_url)


/tmp/ipykernel_55/251970998.py:33: DeprecationWarning:

The 'show_api' parameter in launch() will be removed in Gradio 6.0. You will need to use the 'footer_links' parameter instead. To replicate show_api=False, In Gradio 6.0, use footer_links=['gradio', 'settings'].



* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://17c8ed8e76f8c0113c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Local URL: http://127.0.0.1:7860/
Public share URL: https://17c8ed8e76f8c0113c.gradio.live
